# Layer Attribution in BERT: Which Layers Are Most Important?

**Learning objectives:**
- Apply LayerConductance to measure each BERT encoder layer's contribution
- Visualize layer importance profiles for different prediction types
- Compare layer importance for positive vs. negative sentences
- Build a token × layer heatmap showing where token attributions emerge

**Estimated time:** 15 minutes

**Model:** `textattack/bert-base-uncased-SST-2`

In [ ]:
learning_objectives(["- Apply LayerConductance to measure each BERT encoder layer's contribution", "Visualize layer importance profiles for different prediction types", "Compare layer importance for positive vs. negative sentences", "Build a token × layer heatmap showing where token attributions emerge", "15 minutes", "`textattack/bert-base-uncased-SST-2`"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from captum.attr import LayerConductance, LayerIntegratedGradients

model_name = "textattack/bert-base-uncased-SST-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

LABELS = ["NEGATIVE", "POSITIVE"]
N_LAYERS = len(model.bert.encoder.layer)  # 12 for BERT-base
print(f"Model loaded. Encoder layers: {N_LAYERS}")

In [ ]:
def tokenize(text, max_length=128):
    inp = tokenizer(text, return_tensors='pt', max_length=max_length, truncation=True)
    return inp['input_ids'], inp.get('attention_mask'), inp.get('token_type_ids')

def make_baseline(ids):
    return torch.full_like(ids, tokenizer.pad_token_id)

def forward_func(input_ids, attention_mask=None, token_type_ids=None):
    return model(input_ids=input_ids, attention_mask=attention_mask,
                 token_type_ids=token_type_ids).logits

def aggregate_subwords(tokens, attrs, skip_special=True):
    special = {'[CLS]', '[SEP]', '[PAD]'}
    word_tokens, word_attrs = [], []
    cur_word, cur_attr = None, 0.0
    for tok, attr in zip(tokens, attrs):
        if skip_special and tok in special: continue
        if tok.startswith('##'):
            cur_word = (cur_word or '') + tok[2:]; cur_attr += attr
        else:
            if cur_word: word_tokens.append(cur_word); word_attrs.append(cur_attr)
            cur_word, cur_attr = tok, attr
    if cur_word: word_tokens.append(cur_word); word_attrs.append(cur_attr)
    return word_tokens, np.array(word_attrs)

print("Helpers ready.")

## 1. Layer Conductance: One Scalar Per Layer

In [ ]:
def compute_all_layer_conductance(text: str, n_steps: int = 20) -> dict:
    """Compute LayerConductance for embedding layer + all 12 encoder layers."""
    ids, mask, ttype = tokenize(text)
    baseline = make_baseline(ids)

    with torch.no_grad():
        pred = forward_func(ids, mask, ttype).argmax(dim=1).item()

    layer_scores = []
    layer_names  = []

    # Embedding layer
    lc = LayerConductance(forward_func, model.bert.embeddings)
    cond = lc.attribute(
        ids, baseline, additional_forward_args=(mask, ttype),
        target=pred, n_steps=n_steps,
    )
    layer_scores.append(cond.abs().sum().item())
    layer_names.append('Emb')

    # Each encoder layer
    for i, encoder_layer in enumerate(model.bert.encoder.layer):
        lc = LayerConductance(forward_func, encoder_layer)
        cond = lc.attribute(
            ids, baseline, additional_forward_args=(mask, ttype),
            target=pred, n_steps=n_steps,
        )
        layer_scores.append(cond.abs().sum().item())
        layer_names.append(f'L{i}')

    return {
        'text': text,
        'pred_label': LABELS[pred],
        'layer_scores': np.array(layer_scores),
        'layer_names': layer_names,
    }


# Test
sample_text = "The film was absolutely brilliant from start to finish."
print(f"Computing layer conductance for: '{sample_text}'")
layer_result = compute_all_layer_conductance(sample_text)

print(f"Prediction: {layer_result['pred_label']}")
print("\nLayer importance:")
for name, score in zip(layer_result['layer_names'], layer_result['layer_scores']):
    bar = '█' * int(score / layer_result['layer_scores'].max() * 25)
    print(f"  {name:5s}: {score:.4f}  {bar}")

## 2. Layer Importance Bar Chart

In [ ]:
def plot_layer_importance(results_list: list, figsize=(12, 5)):
    """Plot layer importance profiles for multiple sentences."""
    n_layers = len(results_list[0]['layer_names'])
    x = np.arange(n_layers)
    width = 0.8 / len(results_list)

    colors_pos = plt.cm.Greens(np.linspace(0.4, 0.9, sum(1 for r in results_list if r['pred_label']=='POSITIVE')))
    colors_neg = plt.cm.Reds(np.linspace(0.4, 0.9, sum(1 for r in results_list if r['pred_label']=='NEGATIVE')))
    color_iter_pos = iter(colors_pos)
    color_iter_neg = iter(colors_neg)

    fig, ax = plt.subplots(figsize=figsize)

    for i, result in enumerate(results_list):
        # Normalize scores to [0, 1] for fair comparison
        scores = result['layer_scores'] / result['layer_scores'].max()
        color = next(color_iter_pos) if result['pred_label'] == 'POSITIVE' else next(color_iter_neg)

        ax.bar(
            x + i * width - (len(results_list)-1)*width/2,
            scores, width*0.9,
            color=color, alpha=0.85,
            label=f"{result['pred_label']}: '{result['text'][:30]}...'"
        )

    ax.set_xticks(x)
    ax.set_xticklabels(results_list[0]['layer_names'], fontsize=9)
    ax.set_xlabel('BERT Layer')
    ax.set_ylabel('Normalized LayerConductance magnitude')
    ax.set_title('BERT Layer Importance by Sentence\n(SST-2 sentiment classification)',
                 fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    return fig


# Compare multiple sentences
comparison_texts = [
    "The film was absolutely brilliant from start to finish.",
    "A terrible waste of time with no redeeming qualities.",
    "The movie was not boring at all.",
    "An outstanding performance by the entire cast.",
]

print("Computing layer conductance for all comparison texts...")
all_layer_results = [compute_all_layer_conductance(t, n_steps=15) for t in comparison_texts]
print("Done.")

fig = plot_layer_importance(all_layer_results)
plt.tight_layout()
plt.savefig('bert_layer_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bert_layer_importance.png")

## 3. Token × Layer Attribution Heatmap

Show how each token's attribution score evolves across layers.

In [ ]:
def compute_token_layer_heatmap(text: str, n_steps: int = 15) -> dict:
    """Compute per-token attribution at each BERT encoder layer."""
    ids, mask, ttype = tokenize(text)
    baseline = make_baseline(ids)
    tokens = tokenizer.convert_ids_to_tokens(ids[0].tolist())

    with torch.no_grad():
        pred = forward_func(ids, mask, ttype).argmax(dim=1).item()

    layer_token_attrs = []  # list of (seq_len,) arrays, one per layer

    for i, encoder_layer in enumerate(model.bert.encoder.layer):
        lc = LayerConductance(forward_func, encoder_layer)
        cond = lc.attribute(
            ids, baseline, additional_forward_args=(mask, ttype),
            target=pred, n_steps=n_steps,
        )
        # Sum over hidden dim → one score per token
        token_scores = cond.sum(dim=-1).squeeze(0).detach().numpy()
        layer_token_attrs.append(token_scores)

    return {
        'text': text,
        'tokens': tokens,
        'layer_token_attrs': np.array(layer_token_attrs),  # (12, seq_len)
        'pred_label': LABELS[pred],
    }


heatmap_text = "The movie was absolutely brilliant with stellar performances."
print(f"Building token × layer heatmap for: '{heatmap_text}'")
heatmap_result = compute_token_layer_heatmap(heatmap_text)
print(f"Shape: {heatmap_result['layer_token_attrs'].shape} (12 layers × seq_len tokens)")
print(f"Prediction: {heatmap_result['pred_label']}")

In [ ]:
# Filter to non-special tokens for cleaner display
tokens = heatmap_result['tokens']
layer_attrs = heatmap_result['layer_token_attrs']
special = {'[CLS]', '[SEP]', '[PAD]'}
keep_indices = [i for i, t in enumerate(tokens) if t not in special]
display_tokens = [t for t in tokens if t not in special]
display_attrs  = layer_attrs[:, keep_indices]

# Aggregate subwords: for each subword ## merge with predecessor
word_list, col_groups = [], []
cur_word, cur_group = None, []
for i, tok in enumerate(display_tokens):
    if tok.startswith('##'):
        cur_word = (cur_word or '') + tok[2:]
        cur_group.append(i)
    else:
        if cur_word:
            word_list.append(cur_word)
            col_groups.append(cur_group)
        cur_word = tok
        cur_group = [i]
if cur_word:
    word_list.append(cur_word)
    col_groups.append(cur_group)

# Sum subword group columns
word_layer_matrix = np.array([
    display_attrs[:, group].sum(axis=1) for group in col_groups
]).T  # shape: (n_layers, n_words)

# Plot heatmap
fig, ax = plt.subplots(figsize=(max(10, len(word_list)*1.2), 7))
vmax = np.abs(word_layer_matrix).max()
im = ax.imshow(word_layer_matrix, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

ax.set_xticks(range(len(word_list)))
ax.set_xticklabels(word_list, rotation=30, ha='right', fontsize=11)
ax.set_yticks(range(12))
ax.set_yticklabels([f'Layer {i}' for i in range(12)], fontsize=9)
ax.set_xlabel('Word', fontsize=12)
ax.set_ylabel('BERT Encoder Layer', fontsize=12)
ax.set_title(
    f'Token × Layer Attribution Heatmap (LayerConductance)\n'
    f'"{heatmap_text}"\nPrediction: {heatmap_result["pred_label"]}',
    fontsize=12, fontweight='bold'
)
plt.colorbar(im, ax=ax, label='Attribution (red=positive, blue=negative)')

# Add annotations for strongest tokens in each layer
for layer_i in range(12):
    row = word_layer_matrix[layer_i]
    top_idx = np.argmax(np.abs(row))
    ax.text(top_idx, layer_i, '★', ha='center', va='center',
            fontsize=8, color='black', alpha=0.6)

plt.tight_layout()
plt.savefig('bert_token_layer_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bert_token_layer_heatmap.png")
print("★ marks the most important token at each layer.")

## 4. Layer Importance Profile: Positive vs. Negative

In [ ]:
# Compute mean layer importance across positive and negative sentences
positive_texts = [
    "The film was absolutely brilliant from start to finish.",
    "An outstanding and moving cinematic experience.",
    "Stunning visuals and powerful performances throughout.",
    "A masterfully crafted story that resonates deeply.",
]

negative_texts = [
    "A terrible waste of time with no redeeming qualities.",
    "The acting was abysmal and the plot incoherent.",
    "Painfully dull and devoid of any originality.",
    "A disappointingly shallow and forgettable film.",
]

print("Computing layer profiles for positive and negative sentences...")
pos_profiles = [compute_all_layer_conductance(t, n_steps=12) for t in positive_texts]
neg_profiles = [compute_all_layer_conductance(t, n_steps=12) for t in negative_texts]

# Average and normalize
pos_mean = np.array([r['layer_scores'] for r in pos_profiles]).mean(axis=0)
neg_mean = np.array([r['layer_scores'] for r in neg_profiles]).mean(axis=0)
pos_mean /= pos_mean.max()
neg_mean /= neg_mean.max()

layer_names = pos_profiles[0]['layer_names']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(layer_names))
width = 0.38

ax.bar(x - width/2, pos_mean, width, label=f'POSITIVE (n={len(pos_profiles)})',
       color='#2ca25f', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, neg_mean, width, label=f'NEGATIVE (n={len(neg_profiles)})',
       color='#d73027', alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(layer_names, fontsize=9)
ax.set_xlabel('BERT Layer')
ax.set_ylabel('Normalized LayerConductance (mean)')
ax.set_title('Layer Importance Profile: POSITIVE vs. NEGATIVE Sentences\n'
             '(SST-2 sentiment, BERT-base-uncased)',
             fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('bert_layer_pos_neg.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: bert_layer_pos_neg.png")

# Top 3 layers for each class
top3_pos = sorted(range(len(pos_mean)), key=lambda i: -pos_mean[i])[:3]
top3_neg = sorted(range(len(neg_mean)), key=lambda i: -neg_mean[i])[:3]
print(f"\nTop-3 layers for POSITIVE: {[layer_names[i] for i in top3_pos]}")
print(f"Top-3 layers for NEGATIVE: {[layer_names[i] for i in top3_neg]}")

## Self-Check Questions

1. **Last-layer dominance:** For sentiment classification, later layers should dominate. Does your layer importance profile show this? If early layers also show high importance, what might this indicate?

2. **Positive vs. negative pattern:** Do positive and negative sentences show similar layer importance profiles, or do they differ? What would a difference suggest about how BERT processes positive vs. negative sentiment?

3. **Token × layer heatmap:** In the heatmap, do sentiment-bearing words ("brilliant", "stellar") have consistently high attribution across all layers, or does their importance emerge at specific layers? What layer does attribution for "brilliant" first become notable?

4. **n_steps=15 vs. n_steps=50:** We used fewer steps here to save time. Does reducing n_steps affect the rank order of layers, or just the absolute values? Run with both settings and compare.

**Exercise:** Repeat the layer analysis for a sentence with negation ("The movie was NOT terrible"). Does the layer importance profile differ from a straightforwardly positive sentence? Can you see where in the layer stack the negation is processed?